In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    cross_val_predict,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
)

In [ ]:
url = "https://raw.githubusercontent.com/Ankit152/IMDB-Sentiment-Analysis/master/IMDB-Dataset.csv"
df = pd.read_csv(url).sample(10000, random_state=42) # Subset for speed

# Encode target
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(f"Dataset loaded with {df.shape[0]} reviews.")
df.head()

In [ ]:
def plot_confusion_matrix_simple(cm, labels=("Negative", "Positive")):
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title("Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.show()


def plot_decision_boundary(model, X, y, ax, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25)
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", s=35)
    ax.set_title(title)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")


def summarize_grid(grid, model_name):
    row = {
        "model": model_name,
        "best_params": grid.best_params_,
        "best_cv_accuracy": round(grid.best_score_, 4),
    }
    return row

In [ ]:
X = df["review"].astype(str)
y = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer_baseline = CountVectorizer(max_features=1000)  #Bag Of words -> 1_000 features
X_train_baseline_bow = vectorizer_baseline.fit_transform(X_train)

X_train_bow_df = pd.DataFrame(
    X_train_baseline_bow.toarray(),
    columns=vectorizer_baseline.get_feature_names_out(),
    index=X_train.index
)

print(f"Shape of X_train_vectorized: {X_train_baseline_bow.shape}")

model = LogisticRegression(max_iter=1000, random_state=42) # Increased max_iter for convergence
model.fit(X_train_baseline_bow, y_train)

X_test_baseline_bow = vectorizer_baseline.transform(X_test)
y_pred_baseline = model.predict(X_test_baseline_bow)

print("Classification Report:")
print(f"F1 Score: {f1_score(y_test, y_pred_baseline)}")

In [ ]:
depths = range(1, 21)

train_scores = []
cv_scores = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)

    scores = cross_validate(
        tree,
        X_train_baseline_bow,
        y_train,
        cv=5,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1
    )

    train_scores.append(scores["train_score"].mean())
    cv_scores.append(scores["test_score"].mean())

plt.figure(figsize=(8, 5))
plt.plot(depths, train_scores, marker="o", label="Train accuracy")
plt.plot(depths, cv_scores, marker="o", label="CV accuracy")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Bias / Variance with Decision Trees")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#### SVM


In [ ]:
svm_configs = [
    {"C": 0.1, "gamma": 0.1},
    {"C": 10,  "gamma": 0.1},
    {"C": 0.1, "gamma": 10},
    {"C": 10,  "gamma": 10},
]

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.ravel()

for ax, cfg in zip(axes, svm_configs):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"]))  #RBF -> polynomial kernel → non-linear boundary
    ])
    model.fit(X_train_bow_df.values, y_train)
    plot_decision_boundary(
        model,
        X_train_bow_df,
        y_train,
        ax,
        title=f"C={cfg['C']} | gamma={cfg['gamma']}"
    )

plt.tight_layout()
plt.show()

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", probability=True, random_state=42))
    ]),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

comparison_rows = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train_bow_df,
        y_train,
        cv=cv,
        scoring=["accuracy", "roc_auc"],
        n_jobs=-1
    )

    comparison_rows.append({
        "model": name,
        "mean_cv_accuracy": scores["test_accuracy"].mean(),
        "std_cv_accuracy": scores["test_accuracy"].std(),
        "mean_cv_roc_auc": scores["test_roc_auc"].mean(),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("mean_cv_accuracy", ascending=False)
comparison_df

,model,mean_cv_accuracy,std_cv_accuracy,mean_cv_roc_auc
1,SVM,0.853625,0.008491,0.925458
2,Gradient Boosting,0.810250,0.012315,0.894506
0,Decision Tree,0.679875,0.011986,0.679858
